
# Notebook eksperimen perbandingan preprocessing A/B/C untuk deteksi komentar judol

Notebook ini dipakai untuk membandingkan tiga metode preprocessing secara **fair** dengan protokol:

1. Split raw yang sama persis: `train_raw`, `val_raw`, `test_raw`
2. Model dan hyperparameter yang sama
3. Preprocessing yang berbeda hanya di fungsi `preprocess_a`, `preprocess_b`, `preprocess_c`
4. Threshold dicari dari **validation set**
5. **Test set hanya dipakai untuk preprocessing terbaik**

Notebook ini cocok dijalankan di **Google Colab GPU**.



## Cara pakai singkat

1. Upload tiga file CSV raw:
   - `train_raw.csv`
   - `val_raw.csv`
   - `test_raw.csv`
2. Pastikan kolom:
   - `text`
   - `label`
3. Isi fungsi:
   - `preprocess_a`
   - `preprocess_b`
   - `preprocess_c`
4. Jalankan eksperimen per preprocessing
5. Lihat tabel hasil validation
6. Pilih winner
7. Jalankan evaluasi test final untuk winner saja


In [ ]:

# Uncomment jika perlu di Colab
# !pip -q install transformers datasets accelerate scikit-learn pandas numpy matplotlib

import os
import re
import gc
import json
import math
import random
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

# =========================
# CONFIG
# =========================

TRAIN_PATH = "/content/train_raw.csv"
VAL_PATH   = "/content/val_raw.csv"
TEST_PATH  = "/content/test_raw.csv"

TEXT_COL = "text"
LABEL_COL = "label"

BASE_MODEL_NAME = "indobenchmark/indobert-base-p1"
NUM_LABELS = 2
MAX_LENGTH = 128

# Samakan untuk semua engineer
SEEDS = [42]      # Bisa diganti jadi [42, 123, 777] kalau mau lebih robust
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

# Rule threshold selection
THRESHOLD_START = 0.10
THRESHOLD_END = 0.91
THRESHOLD_STEP = 0.05
MIN_PRECISION_FLOOR = 0.60   # boleh diganti, mis. 0.65 / 0.70
PRIMARY_OBJECTIVE = "recall" # tetap recall-first

# Output
OUTPUT_ROOT = "/content/outputs_preprocess_compare"
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

# Hanya ubah ke True saat sudah memilih winner dari validation
RUN_FINAL_TEST = False
WINNER_PREPROCESS_NAME = None  # contoh: "A" / "B" / "C"
WINNER_THRESHOLD = None        # isi setelah winner dipilih

print("Output root:", OUTPUT_ROOT)


In [ ]:

# =========================
# LOAD RAW SPLITS
# =========================

train_raw = pd.read_csv(TRAIN_PATH)
val_raw = pd.read_csv(VAL_PATH)
test_raw = pd.read_csv(TEST_PATH)

required_cols = {TEXT_COL, LABEL_COL}
for name, df in {
    "train_raw": train_raw,
    "val_raw": val_raw,
    "test_raw": test_raw,
}.items():
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

print("train_raw:", train_raw.shape)
print("val_raw  :", val_raw.shape)
print("test_raw :", test_raw.shape)

display(train_raw.head(3))



## PII masking

PII diproses **sebelum** preprocessing utama, supaya:
- data lebih aman
- model tidak belajar pola sensitif yang tidak relevan


In [ ]:

# =========================
# PII MASKING
# =========================

PHONE_PATTERN = re.compile(r'(\+62|62|0)8[1-9][0-9]{6,10}')
EMAIL_PATTERN = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')
NIK_PATTERN = re.compile(r'\b\d{16}\b')

def mask_pii(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = PHONE_PATTERN.sub("[PHONE]", text)
    text = EMAIL_PATTERN.sub("[EMAIL]", text)
    text = NIK_PATTERN.sub("[NIK]", text)
    return text



## Tempat fungsi preprocessing masing-masing engineer

Yang dibedakan antarmetode **hanya bagian ini**.  
Bagian training, tokenizer, hyperparameter, dan evaluasi harus sama.


In [ ]:

# =========================
# PREPROCESSING A / B / C
# =========================

def normalize_unicode_noise(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200b-\u200f\u202a-\u202e\u2060-\u206f\ufeff]", " ", text)
    text = re.sub(r"[▀▄▌▐■□●◆◇◼◻❚█]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# -------------------------------------------------
# Engineer A
# -------------------------------------------------
def preprocess_a(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = mask_pii(text)
    text = normalize_unicode_noise(text)

    # TODO: paste logika preprocessing Engineer A di sini
    # contoh baseline sederhana:
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

# -------------------------------------------------
# Engineer B
# -------------------------------------------------
def preprocess_b(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = mask_pii(text)
    text = normalize_unicode_noise(text)

    # TODO: paste logika preprocessing Engineer B di sini
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

# -------------------------------------------------
# Engineer C
# -------------------------------------------------
def preprocess_c(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = mask_pii(text)
    text = normalize_unicode_noise(text)

    # TODO: paste logika preprocessing Engineer C di sini
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

PREPROCESSING_METHODS = {
    "A": preprocess_a,
    "B": preprocess_b,
    "C": preprocess_c,
}


In [ ]:

# Quick sanity check
samples = [
    "MAIN H.O.K.I sekarang!!!",
    "S L O T gacor banget",
    "hubungi 081234567890 atau email abc@test.com",
    "alexis77 anti rungkat 😈",
]

for name, fn in PREPROCESSING_METHODS.items():
    print(f"\n=== {name} ===")
    for s in samples:
        print("IN :", s)
        print("OUT:", fn(s))


In [ ]:

# =========================
# TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: "labels"}), preserve_index=False)
    return ds

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


In [ ]:

# =========================
# HELPERS
# =========================

def apply_preprocessing(df: pd.DataFrame, preprocess_fn) -> pd.DataFrame:
    out = df.copy()
    out[TEXT_COL] = out[TEXT_COL].fillna("").astype(str).apply(preprocess_fn)
    return out

def get_positive_probs(trainer: Trainer, tokenized_ds: Dataset) -> np.ndarray:
    preds = trainer.predict(tokenized_ds)
    logits = preds.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    return probs

def threshold_sweep(y_true, prob_1, start=THRESHOLD_START, end=THRESHOLD_END, step=THRESHOLD_STEP):
    thresholds = np.arange(start, end, step)
    rows = []
    for th in thresholds:
        y_pred = (prob_1 >= th).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0
        )
        acc = accuracy_score(y_true, y_pred)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        rows.append({
            "threshold": round(float(th), 4),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "accuracy": float(acc),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        })
    return pd.DataFrame(rows)

def pick_best_threshold(sweep_df: pd.DataFrame, min_precision_floor=MIN_PRECISION_FLOOR) -> pd.Series:
    candidates = sweep_df[sweep_df["precision"] >= min_precision_floor].copy()

    if len(candidates) == 0:
        # fallback: kalau tidak ada yang lolos precision floor, ambil recall tertinggi lalu f1 tertinggi
        candidates = sweep_df.copy()

    candidates = candidates.sort_values(
        by=["recall", "f1", "precision", "threshold"],
        ascending=[False, False, False, True]
    )
    return candidates.iloc[0]

def build_compute_metrics_default(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


In [ ]:

# =========================
# TRAIN ONE EXPERIMENT
# =========================

def train_one_preprocessing(preprocess_name: str, preprocess_fn, seed: int):
    set_seed(seed)
    run_dir = Path(OUTPUT_ROOT) / f"preprocess_{preprocess_name}_seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    # 1) apply preprocessing
    train_df = apply_preprocessing(train_raw, preprocess_fn)
    val_df = apply_preprocessing(val_raw, preprocess_fn)

    # 2) dataset
    train_ds = make_hf_dataset(train_df).map(tokenize_batch, batched=True)
    val_ds = make_hf_dataset(val_df).map(tokenize_batch, batched=True)

    keep_cols = ["input_ids", "attention_mask", "labels"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
    val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])

    # 3) model
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        num_labels=NUM_LABELS,
    )

    # 4) args
    args = TrainingArguments(
        output_dir=str(run_dir),
        overwrite_output_dir=True,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_recall",
        greater_is_better=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=build_compute_metrics_default,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    # 5) train
    trainer.train()

    # 6) validation probabilities
    val_prob_1 = get_positive_probs(trainer, val_ds)
    val_labels = np.array(val_df[LABEL_COL].tolist())

    # 7) threshold sweep on validation
    sweep_df = threshold_sweep(val_labels, val_prob_1)
    best_row = pick_best_threshold(sweep_df)

    # save artifacts
    sweep_path = run_dir / "threshold_sweep_val.csv"
    sweep_df.to_csv(sweep_path, index=False)

    trainer.save_model(str(run_dir / "best_model"))
    tokenizer.save_pretrained(str(run_dir / "best_model"))

    result = {
        "preprocessing": preprocess_name,
        "seed": seed,
        "best_threshold": float(best_row["threshold"]),
        "val_precision": float(best_row["precision"]),
        "val_recall": float(best_row["recall"]),
        "val_f1": float(best_row["f1"]),
        "val_accuracy": float(best_row["accuracy"]),
        "val_tn": int(best_row["tn"]),
        "val_fp": int(best_row["fp"]),
        "val_fn": int(best_row["fn"]),
        "val_tp": int(best_row["tp"]),
        "model_dir": str(run_dir / "best_model"),
        "sweep_csv": str(sweep_path),
    }
    return result



## Jalankan eksperimen validation untuk semua preprocessing

Kalau resource Colab terbatas, jalankan dulu dengan:
- `SEEDS = [42]`

Kalau sudah stabil dan ingin lebih kuat:
- `SEEDS = [42, 123, 777]`


In [ ]:

# =========================
# RUN ALL VALIDATION EXPERIMENTS
# =========================

all_results = []

for preprocess_name, preprocess_fn in PREPROCESSING_METHODS.items():
    for seed in SEEDS:
        print(f"\n===== Running preprocess {preprocess_name} | seed={seed} =====")
        result = train_one_preprocessing(preprocess_name, preprocess_fn, seed)
        all_results.append(result)

        # memory cleanup
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(
    by=["val_recall", "val_f1", "val_precision"],
    ascending=[False, False, False]
).reset_index(drop=True)

results_path = Path(OUTPUT_ROOT) / "validation_leaderboard.csv"
results_df.to_csv(results_path, index=False)

print("Saved:", results_path)
display(results_df)


In [ ]:

# Kalau pakai multi-seed, ini ringkasan per preprocessing
if len(SEEDS) > 1:
    summary_df = (
        results_df.groupby("preprocessing", as_index=False)
        .agg(
            mean_recall=("val_recall", "mean"),
            std_recall=("val_recall", "std"),
            mean_f1=("val_f1", "mean"),
            std_f1=("val_f1", "std"),
            mean_precision=("val_precision", "mean"),
        )
        .sort_values(by=["mean_recall", "mean_f1", "mean_precision"], ascending=[False, False, False])
        .reset_index(drop=True)
    )
    display(summary_df)
else:
    print("Single-seed mode: summary antar-seed dilewati.")



## Cara memilih winner

Prioritas rekomendasi:

1. **Recall validation tertinggi**
2. Jika mirip, pilih **F1** yang lebih tinggi
3. Pastikan precision tidak jatuh terlalu rendah
4. Lihat juga `FN` karena false negative adalah risiko utama

Setelah memilih winner, isi:
- `WINNER_PREPROCESS_NAME`
- `WINNER_THRESHOLD`
- set `RUN_FINAL_TEST = True`


In [ ]:

# Contoh bantu ambil kandidat winner otomatis dari leaderboard
display(results_df[[
    "preprocessing",
    "seed",
    "best_threshold",
    "val_recall",
    "val_f1",
    "val_precision",
    "val_fn",
    "val_fp",
    "model_dir"
]])


In [ ]:

# =========================
# FINAL TEST - JALANKAN HANYA SETELAH WINNER DIPILIH
# =========================

def evaluate_on_test(preprocess_name: str, preprocess_fn, model_dir: str, threshold: float):
    test_df = apply_preprocessing(test_raw, preprocess_fn)

    test_ds = make_hf_dataset(test_df).map(tokenize_batch, batched=True)
    keep_cols = ["input_ids", "attention_mask", "labels"]
    test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_cols])

    test_tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    test_model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True)

    args = TrainingArguments(
        output_dir=str(Path(OUTPUT_ROOT) / "tmp_test_eval"),
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        report_to="none",
    )

    trainer = Trainer(
        model=test_model,
        args=args,
        tokenizer=test_tokenizer,
    )

    prob_1 = get_positive_probs(trainer, test_ds)
    y_true = np.array(test_df[LABEL_COL].tolist())
    y_pred = (prob_1 >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    report = classification_report(y_true, y_pred, digits=4, zero_division=0)
    result = {
        "preprocessing": preprocess_name,
        "threshold": float(threshold),
        "test_accuracy": float(acc),
        "test_precision": float(precision),
        "test_recall": float(recall),
        "test_f1": float(f1),
        "confusion_matrix": cm.tolist(),
        "classification_report": report,
    }
    return result

if RUN_FINAL_TEST:
    assert WINNER_PREPROCESS_NAME in PREPROCESSING_METHODS, "Isi WINNER_PREPROCESS_NAME dengan A/B/C"
    assert WINNER_THRESHOLD is not None, "Isi WINNER_THRESHOLD dulu"

    # Ambil model terbaik untuk winner dari leaderboard
    winner_row = (
        results_df[results_df["preprocessing"] == WINNER_PREPROCESS_NAME]
        .sort_values(by=["val_recall", "val_f1", "val_precision"], ascending=[False, False, False])
        .iloc[0]
    )

    final_result = evaluate_on_test(
        preprocess_name=WINNER_PREPROCESS_NAME,
        preprocess_fn=PREPROCESSING_METHODS[WINNER_PREPROCESS_NAME],
        model_dir=winner_row["model_dir"],
        threshold=WINNER_THRESHOLD,
    )

    final_path = Path(OUTPUT_ROOT) / "final_test_result.json"
    with open(final_path, "w") as f:
        json.dump(final_result, f, indent=2)

    print("FINAL TEST RESULT")
    print(json.dumps(final_result, indent=2))
    print("\nClassification report:")
    print(final_result["classification_report"])
    print("\nSaved:", final_path)

else:
    print("RUN_FINAL_TEST masih False. Test final belum dijalankan.")



## Catatan penting

- Jangan ubah hyperparameter antar-preprocessing
- Jangan pakai test set untuk pilih threshold
- Jangan pakai test set untuk pilih winner
- Kalau mau lebih kuat, tambahkan evaluasi per kategori obfuscation
- Untuk inference production, gunakan threshold winner, **jangan argmax default**
